# Conceptos Fundamentales — LLMs y Generative AI

**Módulo 3 — Introducción a AI Engineering** | DSRP Machine Learning Engineering

---

> 📖 Este notebook es un **glosario de referencia** para los conceptos fundamentales de LLMs, Transformers y GenAI. Cada concepto incluye definición, matemática, visualización y código.

### Índice

**Fundamentos de Transformers:**
1. [Tokens y Tokenización](#1)
2. [Embeddings — representación vectorial](#2)
3. [Similitud coseno y distancia euclidiana](#3)
4. [Softmax — de logits a probabilidades](#4)
5. [Positional Encodings — codificar el orden](#5)
6. [Attention Mechanism — Query, Key, Value](#6)
7. [Multi-Head Attention](#7)
8. [Feed-Forward Networks (FFN)](#8)
9. [Layer Normalization y conexiones residuales](#9)

**Generación y Evaluación:**
10. [Autoregresión — generación token a token](#10)
11. [Sampling strategies — temperature, top-k, top-p](#11)
12. [Perplexity — métrica de evaluación](#12)
13. [Context Window y KV Cache](#13)

**Fine-tuning y Scaling:**
14. [Fine-tuning vs Prompting](#14)
15. [Scaling Laws — más grande = mejor](#15)

**APIs y Producción:**
16. [APIs REST y HTTP — consumir modelos como servicio](#16)
17. [Métricas de similitud vectorial — comparando embeddings](#17)
18. [Vector Databases — búsqueda eficiente de embeddings](#18)
19. [Caching — evitar trabajo redundante](#19)

**Prompt y Context Engineering:**
20. [Chain-of-Thought (CoT) — razonamiento paso a paso](#20)
21. [Context Engineering — orquestar la ventana de contexto](#21)

**Hardware y Optimización:**
22. [ASICs para inferencia — hardware especializado vs GPUs](#22)

**Arquitecturas y Patrones:**
23. [RAG (Retrieval-Augmented Generation) — LLMs con conocimiento externo](#23)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine, euclidean
from scipy.special import softmax

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
np.set_printoptions(precision=3, suppress=True)


---
<a id='1'></a>
## 1. Tokens y Tokenización

**Definición:** La **tokenización** es el proceso de dividir texto en unidades pequeñas llamadas **tokens**. Un token puede ser una palabra completa, una subpalabra, un carácter o incluso un byte.

### ¿Por qué no usar palabras completas?

| Problema | Solución con subpalabras |
|---|---|
| Vocabulario gigante (millones de palabras) | Vocabulario manejable (30k-200k tokens) |
| Palabras fuera del vocabulario (OOV) | Cualquier palabra se puede representar |
| No maneja morfología (correr, corriendo, corrió) | Comparte raíz: `[corr][iendo]`, `[corr][ió]` |

### Algoritmos principales

**1. Byte-Pair Encoding (BPE)** — usado por GPT, LLaMA
- Empieza con caracteres individuales
- Iterativamente fusiona los pares más frecuentes
- Ejemplo: `"running"` → `["run", "ning"]` o `["runn", "ing"]` según frecuencia

**2. WordPiece** — usado por BERT
- Similar a BPE pero maximiza la verosimilitud del corpus
- Usa `##` para marcar continuaciones: `["run", "##ning"]`

**3. SentencePiece** — usado por T5, LLaMA
- Trata el texto como secuencia de bytes (no requiere pre-tokenización)
- Soporta cualquier idioma sin espacios (chino, japonés)

### Propiedades importantes

- **Vocabulario fijo:** típicamente 30k-200k tokens
- **Reversible:** tokens → texto original (casi siempre)
- **Dependiente del modelo:** cada modelo tiene su propio tokenizer
- **Impacto en costo:** más tokens = más caro (APIs cobran por token)

In [ ]:
# Simulación de tokenización BPE simplificada
texto = "running runner run"
print(f"Texto original: '{texto}'\\n")

# Paso 1: caracteres individuales
tokens_iniciales = list(texto.replace(" ", "_"))
print(f"Paso 1 — caracteres: {tokens_iniciales}\\n")

# Paso 2: fusionar pares frecuentes (simulado manualmente)
# En BPE real, se cuenta frecuencia y se fusiona iterativamente
vocab_bpe = {
    "r": "r", "u": "u", "n": "n", "i": "i", "g": "g", "e": "e",
    "run": "run", "ning": "ning", "ner": "ner", "_": "_",
}

# Tokenización final (simulada)
tokens_finales = ["run", "ning", "_", "run", "ner", "_", "run"]
print(f"Paso 2 — después de fusiones BPE: {tokens_finales}\\n")

# Cada token tiene un ID en el vocabulario
vocab_ids = {tok: i for i, tok in enumerate(sorted(vocab_bpe.keys()))}
ids = [vocab_ids[t] for t in tokens_finales]
print(f"IDs en el vocabulario: {ids}")
print(f"\\nVocabulario completo ({len(vocab_ids)} tokens): {vocab_ids}")

# Visualización: comparación de tamaños de vocabulario
fig, ax = plt.subplots(figsize=(8, 4))
metodos = ['Palabras\\ncompletas', 'Caracteres', 'BPE/WordPiece\\n(subpalabras)']
tamaños = [500000, 256, 50000]
colores = ['tomato', 'steelblue', 'green']
bars = ax.barh(metodos, tamaños, color=colores, alpha=0.8)
for bar, tam in zip(bars, tamaños):
    ax.text(tam + 10000, bar.get_y() + bar.get_height()/2,
            f'{tam:,}', va='center', fontweight='bold')
ax.set_xlabel('Tamaño del vocabulario')
ax.set_title('Comparación de métodos de tokenización')
ax.set_xscale('log')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()


---
<a id='16'></a>
## 16. APIs REST y HTTP — consumir modelos como servicio

**Definición:** Una **API (Application Programming Interface)** es una interfaz que permite que dos aplicaciones se comuniquen. **REST (Representational State Transfer)** es un estilo arquitectónico para diseñar APIs sobre HTTP.

### Componentes de una petición HTTP

```http
POST /v1/chat/completions HTTP/1.1
Host: api.openai.com
Authorization: Bearer sk-...
Content-Type: application/json

{"model": "gpt-4o-mini", "messages": [...]}
```

| Componente | Qué es | Ejemplo |
|---|---|---|
| **Método** | Acción a realizar | `GET`, `POST`, `PUT`, `DELETE` |
| **Path** | Recurso específico | `/v1/chat/completions` |
| **Headers** | Metadata | `Authorization`, `Content-Type` |
| **Body** | Datos enviados | JSON con parámetros |

### Métodos HTTP principales

| Método | Uso | Idempotente | Ejemplo |
|---|---|---|---|
| **GET** | Leer datos | ✅ | Obtener lista de modelos |
| **POST** | Crear/ejecutar | ❌ | Generar texto |
| **PUT** | Actualizar completo | ✅ | Actualizar configuración |
| **DELETE** | Eliminar | ✅ | Borrar un fine-tune |

### Códigos de estado HTTP

| Rango | Significado | Ejemplos |
|---|---|---|
| **2xx** | Éxito | 200 OK, 201 Created |
| **4xx** | Error del cliente | 400 Bad Request, 401 Unauthorized, 429 Too Many Requests |
| **5xx** | Error del servidor | 500 Internal Server Error, 503 Service Unavailable |

### JSON — formato de intercambio de datos

**JSON (JavaScript Object Notation)** es el formato estándar para enviar y recibir datos en APIs REST.

```json
{
  "model": "gpt-4o-mini",
  "messages": [
    {"role": "user", "content": "Hola"}
  ],
  "temperature": 0.7,
  "max_tokens": 100
}
```

**Tipos de datos en JSON:**
- `string`: `"texto"`
- `number`: `42`, `3.14`
- `boolean`: `true`, `false`
- `null`: `null`
- `array`: `[1, 2, 3]`
- `object`: `{"key": "value"}`

### Rate Limiting — control de tasa

Los proveedores limitan cuántas peticiones puedes hacer por minuto/hora para evitar abuso:

```
x-ratelimit-limit-requests: 10000       ← máximo por minuto
x-ratelimit-remaining-requests: 9999    ← cuántas te quedan
x-ratelimit-reset-requests: 1704067260  ← cuándo se resetea (Unix timestamp)
```

**Estrategia cuando llegas al límite:**
1. Detectar el error `429 Too Many Requests`
2. Leer el header `Retry-After` o `x-ratelimit-reset-requests`
3. Esperar ese tiempo (+ un poco de jitter aleatorio)
4. Reintentar

**Backoff exponencial:**

```python
import time
import random

def retry_with_backoff(fn, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            return fn()
        except RateLimitError:
            if attempt == max_attempts - 1:
                raise
            wait = (2 ** attempt) + random.random()
            time.sleep(wait)
```

In [ ]:
# Simulación de rate limiting y backoff exponencial
import time
import random

class RateLimitSimulator:
    """Simula un servidor con rate limit."""
    def __init__(self, max_requests=5, window_seconds=10):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.requests = []
    
    def make_request(self):
        """Intenta hacer una petición. Lanza error si excede el límite."""
        now = time.time()
        # Limpiar requests antiguos fuera de la ventana
        self.requests = [t for t in self.requests if now - t < self.window_seconds]
        
        if len(self.requests) >= self.max_requests:
            raise Exception(f"429 Too Many Requests (límite: {self.max_requests}/{self.window_seconds}s)")
        
        self.requests.append(now)
        return f"✅ Request exitoso ({len(self.requests)}/{self.max_requests})"

# Crear simulador con límite de 5 requests por 10 segundos
server = RateLimitSimulator(max_requests=5, window_seconds=10)

print("Simulación de rate limiting:\\n")
print(f"Límite: {server.max_requests} requests por {server.window_seconds} segundos\\n")

# Intentar hacer 8 requests rápidamente
for i in range(8):
    try:
        result = server.make_request()
        print(f"Request {i+1}: {result}")
        time.sleep(0.5)  # pequeña pausa entre requests
    except Exception as e:
        print(f"Request {i+1}: ❌ {e}")
        print(f"           → Esperando 2 segundos antes de reintentar...")
        time.sleep(2)
        # Reintentar
        try:
            result = server.make_request()
            print(f"           → Reintento exitoso: {result}")
        except Exception as e2:
            print(f"           → Reintento falló: {e2}")

print("\\n" + "="*70)
print("Visualización de backoff exponencial")
print("="*70)

# Mostrar cómo crece el tiempo de espera
fig, ax = plt.subplots(figsize=(10, 4))
attempts = range(1, 8)
wait_times = [2**i for i in range(len(attempts))]

ax.bar(attempts, wait_times, color='steelblue', alpha=0.8, edgecolor='white')
for i, (att, wait) in enumerate(zip(attempts, wait_times)):
    ax.text(att, wait + 2, f'{wait}s', ha='center', fontweight='bold')

ax.set_xlabel('Número de intento')
ax.set_ylabel('Tiempo de espera (segundos)')
ax.set_title('Backoff exponencial: tiempo de espera crece exponencialmente\\n(2^intento segundos)')
ax.set_xticks(attempts)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\\n💡 Lecciones clave:")
print("   • Rate limits protegen la infraestructura del proveedor")
print("   • Backoff exponencial evita saturar el servidor con reintentos")
print("   • Agregar jitter aleatorio previene 'thundering herd' (muchos clientes")
print("     reintentando al mismo tiempo)")


---
<a id='17'></a>
## 17. Métricas de similitud vectorial — comparando embeddings

Cuando tenemos dos vectores (embeddings), necesitamos medir qué tan "parecidos" son. Hay varias métricas, cada una con propiedades distintas.

### 1. Similitud coseno (la más usada)

Mide el **ángulo** entre dos vectores, ignorando su magnitud:

$$
\text{sim}_{\cos}(a, b) = \frac{a \cdot b}{\|a\| \|b\|} = \cos(\theta)
$$

**Propiedades:**
- Rango: [-1, 1] (en embeddings de texto, típicamente [0, 1])
- **Invariante a escala:** `sim(a, b) = sim(2a, b)` — solo importa la dirección
- **Cuándo usarla:** Casi siempre — es el estándar para embeddings

**Interpretación:**
- 1.0 → vectores idénticos (mismo ángulo)
- 0.9 → muy similares (sinónimos)
- 0.5 → algo relacionados
- 0.0 → ortogonales (sin relación)
- -1.0 → opuestos (antónimos)

### 2. Distancia euclidiana

Mide la **distancia en línea recta** entre dos puntos:

$$
d_{\text{eucl}}(a, b) = \sqrt{\sum_{i=1}^d (a_i - b_i)^2} = \|a - b\|
$$

**Propiedades:**
- Rango: [0, ∞)
- **Sensible a magnitud:** `d(a, b) ≠ d(2a, 2b)`
- **Cuándo usarla:** Cuando la magnitud del vector tiene significado (raro en embeddings)

**Conversión a similitud:**

$$
\text{sim}_{\text{eucl}}(a, b) = \frac{1}{1 + d_{\text{eucl}}(a, b)}
$$

### 3. Producto punto (dot product)

Combina similitud angular y magnitud:

$$
\text{sim}_{\text{dot}}(a, b) = a \cdot b = \sum_{i=1}^d a_i b_i
$$

**Propiedades:**
- Rango: [-∞, ∞]
- **Más rápido** que coseno (no requiere normalización)
- **Cuándo usarlo:** Cuando los embeddings ya están normalizados (magnitud = 1)

**Relación con coseno:**

Si $\|a\| = \|b\| = 1$ (vectores normalizados), entonces:

$$
\text{sim}_{\text{dot}}(a, b) = \text{sim}_{\cos}(a, b)
$$

### 4. Distancia de Manhattan (L1)

Suma de diferencias absolutas:

$$
d_{\text{L1}}(a, b) = \sum_{i=1}^d |a_i - b_i|
$$

**Cuándo usarla:** Rara vez en embeddings — más común en features categóricas.

### Comparación práctica

| Métrica | Velocidad | Invariante a escala | Uso en embeddings |
|---|---|---|---|
| **Coseno** | Media | ✅ | ⭐⭐⭐⭐⭐ (default) |
| **Euclidiana** | Media | ❌ | ⭐⭐ (ocasional) |
| **Dot product** | Rápida | ❌ | ⭐⭐⭐⭐ (si normalizados) |
| **Manhattan** | Media | ❌ | ⭐ (raro) |

### Normalización de vectores

Para convertir un vector a magnitud 1:

$$
\hat{a} = \frac{a}{\|a\|}
$$

Después de normalizar, `dot(a, b) = cosine(a, b)`, lo cual acelera cálculos.

In [ ]:
# Comparación de métricas de similitud
from scipy.spatial.distance import cosine as cosine_dist, euclidean

# Tres vectores de ejemplo (3D para visualizar)
a = np.array([1.0, 2.0, 1.0])
b = np.array([1.5, 2.5, 1.2])  # similar a 'a'
c = np.array([0.5, 0.2, 0.1])  # diferente de 'a'

print("Vectores de ejemplo:")
print(f"a = {a}")
print(f"b = {b}  (similar a 'a')")
print(f"c = {c}  (diferente de 'a')\\n")

# Calcular todas las métricas
def calcular_metricas(v1, v2, nombre1='a', nombre2='b'):
    # Similitud coseno (scipy devuelve distancia, así que restamos de 1)
    sim_cos = 1 - cosine_dist(v1, v2)
    
    # Distancia euclidiana
    dist_eucl = euclidean(v1, v2)
    sim_eucl = 1 / (1 + dist_eucl)  # convertir a similitud
    
    # Producto punto
    dot_prod = np.dot(v1, v2)
    
    # Manhattan
    dist_l1 = np.sum(np.abs(v1 - v2))
    
    print(f"Comparando {nombre1} vs {nombre2}:")
    print(f"  Similitud coseno:     {sim_cos:.4f}  (rango [0,1], más alto = más similar)")
    print(f"  Distancia euclidiana: {dist_eucl:.4f}  (rango [0,∞), más bajo = más similar)")
    print(f"  Similitud euclidiana: {sim_eucl:.4f}  (convertida a [0,1])")
    print(f"  Producto punto:       {dot_prod:.4f}  (rango [-∞,∞))")
    print(f"  Distancia Manhattan:  {dist_l1:.4f}  (rango [0,∞))\\n")
    
    return sim_cos, dist_eucl, dot_prod

print("="*70)
sim_ab_cos, dist_ab_eucl, dot_ab = calcular_metricas(a, b, 'a', 'b')
print("="*70)
sim_ac_cos, dist_ac_eucl, dot_ac = calcular_metricas(a, c, 'a', 'c')
print("="*70)

# Visualización 3D
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 5))

# Subplot 1: Vectores en 3D
ax1 = fig.add_subplot(121, projection='3d')
origin = [0, 0, 0]

# Dibujar vectores
ax1.quiver(*origin, *a, color='blue', arrow_length_ratio=0.1, linewidth=2, label='a')
ax1.quiver(*origin, *b, color='green', arrow_length_ratio=0.1, linewidth=2, label='b (similar)')
ax1.quiver(*origin, *c, color='red', arrow_length_ratio=0.1, linewidth=2, label='c (diferente)')

# Líneas de distancia euclidiana
ax1.plot([a[0], b[0]], [a[1], b[1]], [a[2], b[2]], 'g--', alpha=0.5, label=f'd(a,b)={dist_ab_eucl:.2f}')
ax1.plot([a[0], c[0]], [a[1], c[1]], [a[2], c[2]], 'r--', alpha=0.5, label=f'd(a,c)={dist_ac_eucl:.2f}')

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Vectores en 3D\\nLíneas punteadas = distancia euclidiana')
ax1.legend()

# Subplot 2: Comparación de métricas
ax2 = fig.add_subplot(122)

metricas = ['Coseno\\n(sim)', 'Euclidiana\\n(1/(1+d))', 'Dot\\nProduct']
valores_ab = [sim_ab_cos, 1/(1+dist_ab_eucl), dot_ab/10]  # dot/10 para escalar
valores_ac = [sim_ac_cos, 1/(1+dist_ac_eucl), dot_ac/10]

x = np.arange(len(metricas))
width = 0.35

bars1 = ax2.bar(x - width/2, valores_ab, width, label='a vs b (similar)', color='green', alpha=0.8)
bars2 = ax2.bar(x + width/2, valores_ac, width, label='a vs c (diferente)', color='red', alpha=0.8)

ax2.set_ylabel('Valor (normalizado)')
ax2.set_title('Comparación de métricas\\n(valores más altos = más similares)')
ax2.set_xticks(x)
ax2.set_xticklabels(metricas)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Anotar valores
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print("\\n💡 Observaciones:")
print("   • Coseno captura bien la similitud angular (b está cerca de a)")
print("   • Euclidiana es sensible a la magnitud (c está muy lejos de a)")
print("   • Dot product combina ángulo y magnitud")
print("   • Para embeddings, COSENO es casi siempre la mejor opción")


---
<a id='18'></a>
## 18. Vector Databases — búsqueda eficiente de embeddings

**Definición:** Una **vector database** es una base de datos optimizada para almacenar y buscar vectores de alta dimensionalidad (embeddings). Permite encontrar los **k vectores más cercanos** (k-NN) a un query en milisegundos, incluso con millones de vectores.

### El problema: búsqueda naive es lenta

Con búsqueda exhaustiva (comparar el query contra todos los vectores):

$$
\text{Complejidad} = O(n \cdot d)
$$

donde $n$ = número de vectores, $d$ = dimensionalidad.

**Ejemplo:** 1 millón de vectores de 1536 dims → 1.5 mil millones de operaciones por query 🐌

### Solución: índices aproximados (ANN)

**ANN (Approximate Nearest Neighbors)** sacrifica un poco de precisión por velocidad:

$$
\text{Complejidad} = O(\log n \cdot d) \text{ o mejor}
$$

**Algoritmos principales:**

| Algoritmo | Cómo funciona | Pros | Contras |
|---|---|---|---|
| **HNSW** | Grafo navegable jerárquico | Muy rápido, alta precisión | Usa mucha RAM |
| **IVF** | Divide el espacio en clusters | Escalable | Requiere entrenamiento |
| **LSH** | Hash sensible a localidad | Simple | Menos preciso |
| **FAISS** | Librería de Meta con múltiples índices | Muy optimizado | Complejo de configurar |

### Vector DBs populares

| Base de datos | Tipo | Cuándo usarla |
|---|---|---|
| **Pinecone** | Managed cloud | Producción, sin infra |
| **Weaviate** | Open-source | Self-hosted, flexible |
| **Qdrant** | Open-source | Rust, muy rápido |
| **Milvus** | Open-source | Escala masiva |
| **ChromaDB** | Open-source | Prototipado local |
| **pgvector** | Extensión PostgreSQL | Ya usas Postgres |

### Operaciones básicas

**1. Insertar vectores:**

```python
# Pseudocódigo
db.insert(
    id='doc_123',
    vector=[0.1, 0.2, ..., 0.5],  # 1536 dims
    metadata={'title': 'Documento X', 'author': 'Y'}
)
```

**2. Buscar similares:**

```python
results = db.search(
    query_vector=[0.15, 0.22, ..., 0.48],
    top_k=5,
    metric='cosine'
)
# Devuelve los 5 vectores más cercanos con sus metadatos
```

**3. Filtrado híbrido:**

```python
# Buscar vectores similares + filtro por metadata
results = db.search(
    query_vector=...,
    top_k=5,
    filter={'author': 'Y', 'year': {'$gte': 2020}}
)
```

### Métricas de performance

**Recall@k:** ¿Qué % de los verdaderos top-k encontramos?

$$
\text{Recall@k} = \frac{\text{# verdaderos positivos en top-k}}{k}
$$

**Latencia:** Tiempo de respuesta (típicamente <10 ms para millones de vectores).

**Throughput:** Queries por segundo (QPS).

### Cuándo NO necesitas una vector DB

- **<10k vectores:** Usa numpy + sklearn (búsqueda exhaustiva es suficiente)
- **Batch processing:** Calcula similitudes offline, guarda en SQL
- **Prototipo rápido:** ChromaDB en memoria

### Cuándo SÍ necesitas una vector DB

- **>100k vectores:** Búsqueda exhaustiva es muy lenta
- **Latencia crítica:** Necesitas respuestas en <100 ms
- **Actualizaciones frecuentes:** Insertar/borrar vectores dinámicamente
- **Producción:** RAG, búsqueda semántica, recomendaciones

In [ ]:
# Comparación: búsqueda naive vs índice optimizado
import time
from sklearn.metrics.pairwise import cosine_similarity

# Simular un corpus de embeddings
np.random.seed(42)
n_docs = 10000
dim = 1536  # dimensión típica de text-embedding-3-small

print(f"Generando {n_docs:,} vectores de {dim} dimensiones...")
corpus_vectors = np.random.randn(n_docs, dim).astype('float32')

# Normalizar para que dot product = cosine similarity
corpus_vectors = corpus_vectors / np.linalg.norm(corpus_vectors, axis=1, keepdims=True)

# Query vector
query_vector = np.random.randn(1, dim).astype('float32')
query_vector = query_vector / np.linalg.norm(query_vector)

print(f"Corpus: {corpus_vectors.shape}")
print(f"Query:  {query_vector.shape}\\n")

# ──────────────────────────────────────────────────────────────────────────────
# Método 1: Búsqueda NAIVE (exhaustiva) con cosine_similarity
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("MÉTODO 1: Búsqueda NAIVE (exhaustiva)")
print("="*70)

start = time.time()
similarities = cosine_similarity(query_vector, corpus_vectors)[0]
top_k_indices = np.argsort(similarities)[::-1][:5]
top_k_scores = similarities[top_k_indices]
time_naive = time.time() - start

print(f"Tiempo: {time_naive*1000:.2f} ms")
print(f"Top-5 índices: {top_k_indices}")
print(f"Top-5 scores:  {top_k_scores}\\n")

# ──────────────────────────────────────────────────────────────────────────────
# Método 2: Índice optimizado con FAISS (si está disponible)
# ──────────────────────────────────────────────────────────────────────────────
try:
    import faiss
    
    print("="*70)
    print("MÉTODO 2: Índice FAISS (optimizado)")
    print("="*70)
    
    # Crear índice FAISS (Flat = exhaustivo pero optimizado con SIMD)
    index = faiss.IndexFlatIP(dim)  # IP = Inner Product (dot product)
    index.add(corpus_vectors)
    
    start = time.time()
    distances, indices = index.search(query_vector, k=5)
    time_faiss = time.time() - start
    
    print(f"Tiempo: {time_faiss*1000:.2f} ms")
    print(f"Top-5 índices: {indices[0]}")
    print(f"Top-5 scores:  {distances[0]}")
    print(f"\\n⚡ Speedup: {time_naive/time_faiss:.1f}x más rápido\\n")
    
    # Verificar que los resultados son idénticos
    assert np.allclose(indices[0], top_k_indices), "Resultados no coinciden"
    print("✅ Verificación: ambos métodos devuelven los mismos resultados\\n")
    
except ImportError:
    print("="*70)
    print("MÉTODO 2: FAISS no disponible")
    print("="*70)
    print("Para instalar: pip install faiss-cpu\\n")
    time_faiss = None

# ──────────────────────────────────────────────────────────────────────────────
# Visualización: escalabilidad
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ANÁLISIS DE ESCALABILIDAD")
print("="*70)

# Simular diferentes tamaños de corpus
corpus_sizes = [1000, 5000, 10000, 50000, 100000]
times_naive = []
times_faiss = []

for size in corpus_sizes:
    # Generar corpus
    vecs = np.random.randn(size, 128).astype('float32')  # dim más chica para velocidad
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    q = np.random.randn(1, 128).astype('float32')
    q = q / np.linalg.norm(q)
    
    # Naive
    start = time.time()
    sims = cosine_similarity(q, vecs)[0]
    _ = np.argsort(sims)[::-1][:5]
    times_naive.append((time.time() - start) * 1000)
    
    # FAISS (si está disponible)
    try:
        import faiss
        idx = faiss.IndexFlatIP(128)
        idx.add(vecs)
        start = time.time()
        _, _ = idx.search(q, k=5)
        times_faiss.append((time.time() - start) * 1000)
    except:
        times_faiss.append(None)

# Graficar
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(corpus_sizes, times_naive, 'o-', label='Naive (exhaustiva)', linewidth=2, markersize=8)
if all(t is not None for t in times_faiss):
    ax.plot(corpus_sizes, times_faiss, 's-', label='FAISS (optimizada)', linewidth=2, markersize=8)

ax.set_xlabel('Tamaño del corpus (# vectores)')
ax.set_ylabel('Tiempo de búsqueda (ms)')
ax.set_title('Escalabilidad: Naive vs FAISS\\n(dim=128, top-k=5)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
ax.set_yscale('log')

# Anotar valores
for i, size in enumerate(corpus_sizes):
    ax.text(size, times_naive[i]*1.2, f'{times_naive[i]:.1f}ms', 
            ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print("\\n💡 Conclusiones:")
print("   • Búsqueda naive: O(n·d) — crece linealmente con el corpus")
print("   • FAISS optimizado: ~10-100x más rápido (SIMD, multithreading)")
print("   • Para >100k vectores, necesitas ANN (HNSW, IVF) para latencia <10ms")
print("   • Vector DBs (Pinecone, Weaviate) usan estos algoritmos internamente")


---
<a id='19'></a>
## 19. Caching — evitar trabajo redundante

**Definición:** Un **cache** es un almacenamiento temporal de resultados de operaciones costosas. Si la misma entrada se repite, devolvemos el resultado guardado en lugar de recalcular.

### ¿Por qué cachear?

| Operación | Costo sin cache | Costo con cache | Ahorro |
|---|---|---|---|
| Llamada a API LLM | $0.00005 + 100ms | 0 + 1ms | 100x velocidad, 100% costo |
| Generar embedding | $0.000001 + 50ms | 0 + 1ms | 50x velocidad, 100% costo |
| Búsqueda en vector DB | 10ms | 1ms | 10x velocidad |

### Tipos de cache

**1. Cache en memoria (in-memory)**

```python
cache = {}  # dict simple

def get_embedding_cached(text):
    if text in cache:
        return cache[text]
    
    embedding = generate_embedding(text)  # operación costosa
    cache[text] = embedding
    return embedding
```

**Pros:** Muy rápido (nanosegundos)  
**Contras:** Se pierde al reiniciar, limitado por RAM

**2. Cache en disco**

```python
import diskcache

cache = diskcache.Cache('./cache_dir')

@cache.memoize()
def get_embedding_cached(text):
    return generate_embedding(text)
```

**Pros:** Persiste entre reinicios  
**Contras:** Más lento que memoria (~1ms)

**3. Cache distribuido (Redis, Memcached)**

```python
import redis

r = redis.Redis(host='localhost', port=6379)

def get_embedding_cached(text):
    key = f'emb:{hash(text)}'
    cached = r.get(key)
    
    if cached:
        return pickle.loads(cached)
    
    embedding = generate_embedding(text)
    r.setex(key, 3600, pickle.dumps(embedding))  # TTL 1 hora
    return embedding
```

**Pros:** Compartido entre múltiples servidores, escalable  
**Contras:** Latencia de red (~1-5ms)

### Estrategias de invalidación

**1. TTL (Time To Live)**

Cada entrada expira después de un tiempo:

```python
cache.set('key', value, ttl=3600)  # expira en 1 hora
```

**2. LRU (Least Recently Used)**

Cuando el cache se llena, borra las entradas menos usadas:

```python
from functools import lru_cache

@lru_cache(maxsize=1000)  # guarda máximo 1000 entradas
def expensive_function(x):
    return x ** 2
```

**3. Invalidación manual**

Borras explícitamente cuando los datos cambian:

```python
cache.delete('user:123:profile')
```

### Cache de prompts (OpenAI, Anthropic)

Los proveedores cachean automáticamente partes del prompt que se repiten:

**OpenAI:** Cachea los primeros ~1024 tokens si se repiten en <5 min (50% descuento)

**Anthropic:** Marca explícitamente qué cachear:

```python
messages = [{
    'role': 'user',
    'content': [
        {'type': 'text', 'text': '<CONTEXTO LARGO>',
         'cache_control': {'type': 'ephemeral'}},  # ← cachea esto
        {'type': 'text', 'text': 'Pregunta específica'}
    ]
}]
```

El contexto cacheado cuesta ~10% del precio normal.

### Métricas de cache

**Hit rate:** % de requests que encuentran el resultado en cache

$$
\text{Hit rate} = \frac{\text{cache hits}}{\text{total requests}} \times 100\%
$$

**Objetivo:** >80% para que valga la pena

**Miss penalty:** Costo adicional de un cache miss (buscar en cache + calcular)

### Cuándo NO cachear

- **Datos que cambian frecuentemente:** Precios en tiempo real, noticias
- **Inputs únicos:** Cada query es diferente (ej. chat sin repetición)
- **Operaciones baratas:** Si calcular cuesta <1ms, el cache agrega overhead

### Cuándo SÍ cachear

- **Operaciones costosas:** APIs, embeddings, búsquedas complejas
- **Inputs repetitivos:** FAQs, búsquedas populares
- **Producción:** Reducir latencia y costo

In [ ]:
# Demostración de caching con diferentes estrategias
import hashlib
import time
from functools import lru_cache

# Simulamos una operación costosa (ej. llamada a API)
def operacion_costosa(input_text):
    """Simula una operación que toma 100ms."""
    time.sleep(0.1)  # 100ms de latencia
    return f"Resultado para: {input_text}"

# ──────────────────────────────────────────────────────────────────────────────
# Estrategia 1: Sin cache (baseline)
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ESTRATEGIA 1: Sin cache")
print("="*70)

queries = ["¿Qué es Python?", "¿Qué es ML?", "¿Qué es Python?", "¿Qué es ML?"]

start = time.time()
for q in queries:
    resultado = operacion_costosa(q)
    print(f"Query: '{q}' → {resultado[:30]}...")
tiempo_sin_cache = time.time() - start

print(f"\nTiempo total: {tiempo_sin_cache:.2f}s")
print(f"Promedio por query: {tiempo_sin_cache/len(queries)*1000:.0f}ms\n")

# ──────────────────────────────────────────────────────────────────────────────
# Estrategia 2: Cache en memoria (dict)
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ESTRATEGIA 2: Cache en memoria (dict)")
print("="*70)

cache_memoria = {}
cache_hits = 0
cache_misses = 0

def operacion_con_cache(input_text):
    global cache_hits, cache_misses
    
    # Usar hash como key (más eficiente que string largo)
    cache_key = hashlib.md5(input_text.encode()).hexdigest()
    
    if cache_key in cache_memoria:
        cache_hits += 1
        print(f"  ✅ CACHE HIT")
        return cache_memoria[cache_key]
    
    cache_misses += 1
    print(f"  ❌ CACHE MISS — calculando...")
    resultado = operacion_costosa(input_text)
    cache_memoria[cache_key] = resultado
    return resultado

start = time.time()
for q in queries:
    print(f"Query: '{q}'")
    resultado = operacion_con_cache(q)
tiempo_con_cache = time.time() - start

hit_rate = cache_hits / len(queries) * 100
print(f"\nTiempo total: {tiempo_con_cache:.2f}s")
print(f"Promedio por query: {tiempo_con_cache/len(queries)*1000:.0f}ms")
print(f"Cache hits: {cache_hits}, misses: {cache_misses}")
print(f"Hit rate: {hit_rate:.0f}%")
print(f"⚡ Speedup: {tiempo_sin_cache/tiempo_con_cache:.1f}x más rápido\n")

# ──────────────────────────────────────────────────────────────────────────────
# Estrategia 3: LRU Cache (con límite de tamaño)
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ESTRATEGIA 3: LRU Cache (límite de 2 entradas)")
print("="*70)

@lru_cache(maxsize=2)
def operacion_lru(input_text):
    print(f"  ❌ CACHE MISS — calculando '{input_text}'...")
    time.sleep(0.1)
    return f"Resultado para: {input_text}"

# Queries que exceden el límite del cache
queries_lru = ["A", "B", "C", "A", "B", "C"]

start = time.time()
for q in queries_lru:
    print(f"Query: '{q}'")
    resultado = operacion_lru(q)
tiempo_lru = time.time() - start

print(f"\nTiempo total: {tiempo_lru:.2f}s")
print(f"Cache info: {operacion_lru.cache_info()}")
print(f"Hit rate: {operacion_lru.cache_info().hits / len(queries_lru) * 100:.0f}%")
print("\n💡 Nota: Con maxsize=2, cuando llega 'C', se borra 'A' (LRU).")
print("   Por eso el segundo 'A' es cache miss.\n")

# ──────────────────────────────────────────────────────────────────────────────
# Visualización: comparación de estrategias
# ──────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Tiempo por estrategia
estrategias = ['Sin cache', 'Con cache\\n(dict)', 'LRU cache\\n(maxsize=2)']
tiempos = [tiempo_sin_cache, tiempo_con_cache, tiempo_lru]
colores = ['tomato', 'green', 'steelblue']

bars = axes[0].bar(estrategias, tiempos, color=colores, alpha=0.8, edgecolor='white')
for bar, t in zip(bars, tiempos):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{t:.2f}s', ha='center', fontweight='bold')

axes[0].set_ylabel('Tiempo total (segundos)')
axes[0].set_title('Comparación de estrategias de caching\\n(4 queries, 2 únicas)')
axes[0].grid(True, alpha=0.3, axis='y')

# Subplot 2: Hit rate
hit_rates = [0, hit_rate, operacion_lru.cache_info().hits / len(queries_lru) * 100]
bars = axes[1].bar(estrategias, hit_rates, color=colores, alpha=0.8, edgecolor='white')
for bar, hr in zip(bars, hit_rates):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{hr:.0f}%', ha='center', fontweight='bold')

axes[1].set_ylabel('Hit rate (%)')
axes[1].set_title('Tasa de aciertos del cache')
axes[1].set_ylim(0, 110)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────────────────────────────────────
# Análisis de ahorro de costo
# ──────────────────────────────────────────────────────────────────────────────
print("="*70)
print("ANÁLISIS DE AHORRO DE COSTO")
print("="*70)

# Supongamos que cada query cuesta $0.0001 (ejemplo)
costo_por_query = 0.0001
queries_por_dia = 10000

# Sin cache
costo_sin_cache = queries_por_dia * costo_por_query

# Con cache (50% hit rate típico)
hit_rate_real = 0.5
queries_reales = queries_por_dia * (1 - hit_rate_real)
costo_con_cache = queries_reales * costo_por_query

ahorro_diario = costo_sin_cache - costo_con_cache
ahorro_mensual = ahorro_diario * 30

print(f"Escenario: {queries_por_dia:,} queries/día, ${costo_por_query:.4f} por query")
print(f"\nSin cache:")
print(f"  Costo diario:  ${costo_sin_cache:.2f}")
print(f"  Costo mensual: ${costo_sin_cache * 30:.2f}")
print(f"\nCon cache (hit rate {hit_rate_real*100:.0f}%):")
print(f"  Costo diario:  ${costo_con_cache:.2f}")
print(f"  Costo mensual: ${costo_con_cache * 30:.2f}")
print(f"\n💰 AHORRO:")
print(f"  Diario:  ${ahorro_diario:.2f} ({ahorro_diario/costo_sin_cache*100:.0f}%)")
print(f"  Mensual: ${ahorro_mensual:.2f}")
print(f"  Anual:   ${ahorro_mensual * 12:.2f}")

print("\n💡 Conclusiones:")
print("   • Cache en memoria: hit rate alto, pero se pierde al reiniciar")
print("   • LRU cache: limita uso de memoria, borra entradas viejas")
print("   • Para producción: Redis/Memcached (persistente, distribuido)")
print("   • Objetivo: hit rate >80% para que valga la pena")
print("   • Ahorro típico: 50-90% en costo y latencia")


---
<a id='20'></a>
## 20. Chain-of-Thought (CoT) — razonamiento paso a paso

**Definición:** **Chain-of-Thought** es una técnica de prompting donde se le pide al modelo que **muestre su razonamiento paso a paso** antes de dar la respuesta final. Mejora dramáticamente la precisión en tareas de razonamiento, matemáticas y lógica.

### ¿Por qué funciona?

Los LLMs son **autoregresivos** — cada token depende de los anteriores. Al generar tokens intermedios de razonamiento, el modelo crea una "cadena" que **guía** la generación de la respuesta correcta.

**Analogía:** Es como "pensar en voz alta" — el modelo se ayuda a sí mismo mostrando los pasos.

### Ejemplo comparativo

**Sin CoT:**
```
Pregunta: Si tengo 5 manzanas y regalo 2, ¿cuántas me quedan?
Respuesta: 7
```
❌ Incorrecto (el modelo se confundió)

**Con CoT:**
```
Pregunta: Si tengo 5 manzanas y regalo 2, ¿cuántas me quedan?
Piensa paso a paso:
Paso 1: Tengo 5 manzanas inicialmente
Paso 2: Regalo 2 manzanas
Paso 3: 5 - 2 = 3
Respuesta: 3 manzanas
```
✅ Correcto

### Variantes de CoT

**1. Zero-shot CoT** — solo agregar "Piensa paso a paso":
```
Pregunta: [problema complejo]
Piensa paso a paso:
```

**2. Few-shot CoT** — dar ejemplos con razonamiento:
```
P: 3 + 2 = ?
R: Paso 1: Tengo 3
    Paso 2: Sumo 2
    Paso 3: 3 + 2 = 5

P: [tu pregunta]
R:
```

**3. Self-Consistency** — generar múltiples razonamientos y votar:
```python
# Generar 5 respuestas con temperature > 0
respuestas = [generar_con_cot() for _ in range(5)]
# Tomar la respuesta más frecuente
respuesta_final = Counter(respuestas).most_common(1)[0][0]
```

### Mejora de precisión (Wei et al., 2022)

| Tarea | Sin CoT | Con CoT | Mejora |
|---|---|---|---|
| Aritmética multi-paso | 18% | 78% | +60 pts |
| Razonamiento simbólico | 32% | 71% | +39 pts |
| Sentido común | 58% | 82% | +24 pts |
| **Promedio** | **36%** | **77%** | **+41 pts** |

### Cuándo usar CoT

✅ **Usar CoT cuando:**
- Tareas de razonamiento matemático
- Lógica multi-paso
- Problemas que requieren planificación
- Debugging de código
- Análisis complejo

❌ **NO usar CoT cuando:**
- Tareas simples (clasificación, extracción)
- Necesitas respuestas muy concisas
- Presupuesto de tokens limitado (CoT usa más tokens)

### Trade-off: precisión vs costo

CoT mejora la precisión pero **usa más tokens**:
- Sin CoT: ~50 tokens de output
- Con CoT: ~200-500 tokens de output

**Costo:** 4-10x más caro por llamada, pero puede valer la pena si evita errores costosos.

In [ ]:
# Simulación de Chain-of-Thought
# (sin llamar a API, solo demostración del concepto)

# Problema de razonamiento
problema = """
Un tren sale de Madrid a las 10:00 viajando a 120 km/h hacia Barcelona (600 km).
Otro tren sale de Barcelona a las 10:30 viajando a 100 km/h hacia Madrid.
¿A qué hora se encuentran?
"""

print("="*70)
print("PROBLEMA DE RAZONAMIENTO")
print("="*70)
print(problema)

print("\n" + "="*70)
print("SOLUCIÓN SIN CoT (respuesta directa)")
print("="*70)
print("Respuesta: 13:00")
print("❌ Probablemente incorrecta (el modelo adivina)")

print("\n" + "="*70)
print("SOLUCIÓN CON CoT (paso a paso)")
print("="*70)

razonamiento_cot = """
Paso 1: Calcular cuánto avanza el tren 1 antes de que salga el tren 2
   - Tren 1 sale a las 10:00
   - Tren 2 sale a las 10:30
   - Diferencia: 30 minutos = 0.5 horas
   - Distancia recorrida por tren 1: 120 km/h × 0.5 h = 60 km

Paso 2: Calcular la distancia restante entre los trenes a las 10:30
   - Distancia total: 600 km
   - Tren 1 ya recorrió: 60 km
   - Distancia restante: 600 - 60 = 540 km

Paso 3: Calcular velocidad de aproximación
   - Tren 1: 120 km/h (hacia Barcelona)
   - Tren 2: 100 km/h (hacia Madrid)
   - Velocidad combinada: 120 + 100 = 220 km/h

Paso 4: Calcular tiempo hasta el encuentro (desde las 10:30)
   - Tiempo = Distancia / Velocidad
   - Tiempo = 540 km / 220 km/h
   - Tiempo ≈ 2.45 horas = 2 horas 27 minutos

Paso 5: Calcular hora del encuentro
   - Hora de referencia: 10:30
   - Sumar 2 horas 27 minutos
   - Hora del encuentro: 12:57 (aproximadamente 13:00)

Respuesta final: Los trenes se encuentran aproximadamente a las 13:00
"""

print(razonamiento_cot)
print("✅ Correcto (el razonamiento guía a la respuesta correcta)")

# Visualización de mejora de precisión
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 5))

categorias = ['Aritmética\\nmulti-paso', 'Razonamiento\\nsimbólico', 
              'Sentido\\ncomún', 'Promedio']
sin_cot = [18, 32, 58, 36]
con_cot = [78, 71, 82, 77]

x = np.arange(len(categorias))
width = 0.35

bars1 = ax.bar(x - width/2, sin_cot, width, label='Sin CoT', 
               color='tomato', alpha=0.8, edgecolor='white')
bars2 = ax.bar(x + width/2, con_cot, width, label='Con CoT', 
               color='green', alpha=0.8, edgecolor='white')

ax.set_ylabel('Precisión (%)')
ax.set_title('Impacto de Chain-of-Thought en precisión\\n(Wei et al., 2022 — PaLM 540B)')
ax.set_xticks(x)
ax.set_xticklabels(categorias)
ax.legend()
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')

# Anotar mejora
for i, (s, c) in enumerate(zip(sin_cot, con_cot)):
    mejora = c - s
    ax.annotate(f'+{mejora}', xy=(i, (s + c) / 2), 
                ha='center', fontsize=10, fontweight='bold', color='blue')

# Anotar valores
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 2,
                f'{int(height)}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n💡 Lecciones clave:")
print("   • CoT mejora +41 puntos porcentuales en promedio")
print("   • Funciona porque el modelo se ayuda a sí mismo con tokens intermedios")
print("   • Trade-off: usa 4-10x más tokens (más caro)")
print("   • Esencial para tareas de razonamiento complejo")


---
<a id='21'></a>
## 21. Context Engineering — orquestar la ventana de contexto

**Definición:** **Context engineering** es el arte de **ensamblar dinámicamente** todo lo que ve el modelo en cada turno: system prompt, ejemplos, documentos recuperados, historial, herramientas, memoria. Va más allá del prompt engineering (escribir instrucciones) — es **seleccionar, comprimir y orquestar** información dentro del presupuesto de tokens.

> _Context engineering es el arte y la ciencia de rellenar la ventana de contexto con la información correcta para el siguiente paso del modelo._  
> — Andrej Karpathy (2024)

### El Context Stack — capas del contexto

```
┌────────────────────────────────────────────┐
│  SYSTEM PROMPT      instrucciones, rol     │  ← fijo
├────────────────────────────────────────────┤
│  TOOL DEFINITIONS   funciones disponibles  │  ← dinámico
├────────────────────────────────────────────┤
│  FEW-SHOT EXAMPLES  ejemplos de referencia │  ← opcional
├────────────────────────────────────────────┤
│  RETRIEVED DOCS     fragmentos de RAG      │  ← dinámico
├────────────────────────────────────────────┤
│  MEMORY / STATE     hechos del usuario     │  ← persistente
├────────────────────────────────────────────┤
│  CONVERSATION       historial de turnos    │  ← crece
├────────────────────────────────────────────┤
│  TOOL RESULTS       outputs de funciones   │  ← dinámico
├────────────────────────────────────────────┤
│  USER MESSAGE       mensaje actual         │  ← nuevo
└────────────────────────────────────────────┘
           ↓
    ventana de contexto (limitada)
```

### Diferencia con Prompt Engineering

| | Prompt Engineering | Context Engineering |
|---|---|---|
| **Foco** | El string de instrucciones | Todo lo que ve el modelo |
| **Naturaleza** | Estático, escrito a mano | Dinámico, ensamblado programáticamente |
| **Incluye** | System + user prompt | System, tools, RAG, memoria, historial |
| **Métrica** | "¿El prompt funciona?" | "¿El contexto es relevante y cabe?" |
| **Skill** | Escribir bien | Seleccionar, comprimir, orquestar |

### Context Budgeting — presupuesto de tokens

Trata cada contexto como un **presupuesto** y reparte entre capas:

```
Presupuesto total: 8,000 tokens

System prompt + tools     :   500 tok  (fijo)
Few-shot examples         :   800 tok
Retrieved docs (top-k)    : 3,000 tok  ← ajustable
Conversation history      : 2,000 tok  ← comprimir si crece
User message              :   200 tok
─────────────────────────────────────
Subtotal input            : 6,500 tok
Reservado para output     : 1,500 tok
─────────────────────────────────────
Total                     : 8,000 tok
```

### Estrategias de compresión

**1. Compactación de historial**

Cuando el historial crece, **resúmelo** en lugar de borrarlo:

```
Turnos 1-20  →  [resumen: usuario preguntó X, respondimos Y...]
Turno 21     →  literal
Turno 22     →  literal
Turno 23     →  literal (actual)
```

**2. Recuperación selectiva (RAG)**

En lugar de mandar documentos completos, manda solo **fragmentos relevantes**:

```
Corpus: 1,000 documentos (10M tokens)
Query: "¿Cómo reseteo mi password?"
         ↓
Embeddings + Vector DB
         ↓
Top-3 fragmentos más relevantes (500 tokens)
         ↓
Inyectar en el contexto
```

**3. Curación de tools**

No registres todas las funciones — solo las **relevantes para este turno**:

```python
# ❌ Malo: 50 tools registradas (5K tokens)
tools = todas_las_funciones()

# ✅ Bueno: solo las relevantes (500 tokens)
tools = seleccionar_tools_relevantes(user_message)
```

### Lost in the Middle — el orden importa

Los modelos prestan más atención al **inicio** y **final** del contexto que al medio (Liu et al., 2023).

**Caída de precisión:** 20-30% cuando la info clave está en el medio vs en los extremos.

**Implicaciones:**
- Pon instrucciones críticas al **inicio Y al final**
- En RAG: ordena chunks con el **más relevante al final** (cerca del user message)
- Si el contexto es muy largo, reorganiza o comprime

### Context Rot — degradación gradual

A medida que el contexto crece, la calidad **degrada** incluso si toda la info está presente:
- Más distractores → modelo se desconcentra
- Más conflictos entre fragmentos
- Peor seguimiento de instrucciones

**Síntoma:** Un prompt que funciona perfecto con 2K tokens empieza a alucinar con 50K.

**Solución:** No es más contexto, es **mejor contexto** — comprime, filtra, prioriza.

### Memoria persistente

Para que un asistente "te conozca" entre sesiones:

```
Memoria (DB externa):
  - Usuario: "Oscar"
  - Preferencias: "respuestas en español", "código con type hints"
  - Proyecto actual: "API REST con FastAPI"
  
         ↓ (recuperar relevante)
         
Context stack:
  [MEMORY] Usuario Oscar prefiere Python con type hints...
  [USER] ¿Cómo hago un endpoint POST?
```

La memoria es otra capa del context stack, gestionada con almacenamiento propio.

In [ ]:
# Simulación de Context Budgeting
import matplotlib.pyplot as plt
import numpy as np

# Definir presupuesto y capas
PRESUPUESTO_TOTAL = 8000
RESERVA_OUTPUT = 1500

capas = {
    'System prompt': 500,
    'Tool definitions': 300,
    'Few-shot examples': 800,
    'Retrieved docs (RAG)': 3000,
    'Conversation history': 2000,
    'User message': 200,
}

total_input = sum(capas.values())
disponible_output = PRESUPUESTO_TOTAL - total_input

print("="*70)
print("CONTEXT BUDGETING — Reparto de tokens por capa")
print("="*70)
print(f"\nPresupuesto total: {PRESUPUESTO_TOTAL:,} tokens\n")

print(f"{'Capa':<25} {'Tokens':>8}   {'% del total':>12}")
print("-"*50)
for capa, tokens in capas.items():
    pct = tokens / PRESUPUESTO_TOTAL * 100
    print(f"{capa:<25} {tokens:>8}   {pct:>10.1f}%")

print("-"*50)
print(f"{'Subtotal input':<25} {total_input:>8}")
print(f"{'Disponible para output':<25} {disponible_output:>8}")
print(f"{'Reserva objetivo':<25} {RESERVA_OUTPUT:>8}")

if disponible_output < RESERVA_OUTPUT:
    print(f"\n⚠️  Solo quedan {disponible_output} tokens para output (objetivo: {RESERVA_OUTPUT})")
    print("   Acción: Recorta una capa (comprime history, baja top-k de RAG)")
else:
    margen = disponible_output - RESERVA_OUTPUT
    print(f"\n✅ Cabe con {margen} tokens de margen")

# Visualización del context stack
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Distribución de tokens por capa
colores = ['#88B5D6', '#E07A5F', '#F4A261', '#2A9D8F', '#E9C46A', '#264653']
valores = list(capas.values())
labels = list(capas.keys())

wedges, texts, autotexts = ax1.pie(valores, labels=labels, autopct='%1.1f%%',
                                     colors=colores, startangle=90)
ax1.set_title('Distribución de tokens por capa del context stack')

# Hacer el texto más legible
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

# Subplot 2: Comparación input vs output
categorias = ['Input\n(context)', 'Output\n(reservado)', 'Margen']
valores_bar = [total_input, RESERVA_OUTPUT, disponible_output - RESERVA_OUTPUT]
colores_bar = ['steelblue', 'tomato', 'green']

bars = ax2.bar(categorias, valores_bar, color=colores_bar, alpha=0.8, edgecolor='white')
ax2.set_ylabel('Tokens')
ax2.set_title('Presupuesto de tokens: Input vs Output')
ax2.set_ylim(0, PRESUPUESTO_TOTAL)
ax2.grid(True, alpha=0.3, axis='y')

# Anotar valores
for bar, val in zip(bars, valores_bar):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 100,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')

# Línea de presupuesto total
ax2.axhline(y=PRESUPUESTO_TOTAL, color='red', linestyle='--', 
            label=f'Presupuesto total: {PRESUPUESTO_TOTAL:,}')
ax2.legend()

plt.tight_layout()
plt.show()

# Simulación de "Lost in the Middle"
print("\n" + "="*70)
print("LOST IN THE MIDDLE — El orden importa")
print("="*70)

# Simular precisión según posición de la info clave
posiciones = ['Inicio', 'Cuarto 1', 'Medio', 'Cuarto 3', 'Final']
precisiones = [85, 75, 55, 70, 82]  # datos de Liu et al., 2023

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(posiciones, precisiones, color='steelblue', alpha=0.8, edgecolor='white')

# Destacar el medio (peor)
bars[2].set_color('tomato')

ax.set_ylabel('Precisión (%)')
ax.set_title('Precisión según posición de la información clave en el contexto\\n(Liu et al., 2023 — "Lost in the Middle")')
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')

# Anotar valores
for bar, prec in zip(bars, precisiones):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{prec}%', ha='center', va='bottom', fontweight='bold')

# Anotar la caída
ax.annotate('', xy=(2, 55), xytext=(0, 85),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(1, 70, '-30%', fontsize=12, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Lecciones clave:")
print("   • Context engineering > prompt engineering para sistemas complejos")
print("   • Trata el contexto como un presupuesto — reparte tokens por capa")
print("   • Comprime historial viejo en lugar de borrarlo")
print("   • Ordena info por relevancia (más relevante al final)")
print("   • Lost in the Middle: -30% precisión cuando info clave está en el medio")
print("   • Más contexto ≠ mejor — combate el context rot")


---
<a id='22'></a>
## 22. ASICs para inferencia — hardware especializado vs GPUs

**Definición:** **ASIC (Application-Specific Integrated Circuit)** es un chip diseñado para una tarea específica, en contraste con GPUs que son de propósito general. Para inferencia de LLMs, ASICs especializados como **LPU (Groq)** y **WSE (Cerebras)** superan a GPUs en velocidad y eficiencia.

### El problema con GPUs para inferencia

Las GPUs fueron diseñadas para **entrenamiento** — operaciones masivas de matriz-matriz. Pero la **inferencia** tiene un perfil diferente:

| Fase | Operación | Bottleneck | GPU utilization |
|---|---|---|---|
| **Prefill** (procesar input) | Matriz-matriz | Cómputo (FLOPS) | Alta (~80%) |
| **Decode** (generar output) | Matriz-vector | **Memoria (bandwidth)** | Baja (~20%) |

En **decode** (generar tokens uno a uno), la GPU está **ociosa** esperando que los datos lleguen desde memoria. Esto se llama **memory-bound inference**.

### Arquitecturas especializadas

**1. Groq LPU (Language Processing Unit)**

Chip ASIC diseñado específicamente para Transformers:

- **Memoria on-chip masiva:** 230 MB de SRAM (vs ~40 MB en GPU)
- **Bandwidth extremo:** 80 TB/s interno (vs ~2 TB/s en GPU)
- **Ejecución determinística:** No hay cache misses — todo es predecible
- **Resultado:** Time-to-first-token ~50 ms, throughput ~280 tok/s (Llama 70B)

**2. Cerebras WSE (Wafer-Scale Engine)**

Un chip del tamaño de una oblea entera:

- **Área:** 46,225 mm² (vs ~826 mm² en H100)
- **Cores:** 850,000 (vs ~17,000 en H100)
- **SRAM on-chip:** 40 GB (vs ~50 MB en H100)
- **Bandwidth:** 20 PB/s interno
- **Resultado:** Throughput >2,000 tok/s (Llama 70B)

### Comparación de arquitecturas

| Métrica | GPU (H100) | Groq LPU | Cerebras WSE |
|---|---|---|---|
| **SRAM on-chip** | 50 MB | 230 MB | 40 GB |
| **Bandwidth** | 3.35 TB/s | 80 TB/s | 20 PB/s |
| **Throughput (Llama 70B)** | ~65 tok/s | ~280 tok/s | ~2,200 tok/s |
| **Time-to-first-token** | 350 ms | 50 ms | 100 ms |
| **Speedup vs GPU** | 1x | 4.3x | 34x |

### ¿Por qué importa?

**Latencia:**
- Chat interactivo: 50 ms vs 350 ms es la diferencia entre "instantáneo" y "esperando"
- Agentes multi-paso: 10 llamadas × 300 ms = 3 segundos de overhead

**Throughput:**
- Generación de reportes: 2,000 tok/s vs 65 tok/s = 30x más rápido
- Batch processing: Procesar 1M tokens en 8 minutos vs 4 horas

**Costo:**
- Menos tiempo de cómputo = menor costo por token
- Mejor utilización del hardware

### ¿Por qué no todos usan ASICs?

**Desventajas:**
- **Costo de desarrollo:** Diseñar un ASIC cuesta $100M-500M
- **Flexibilidad:** Solo sirven para inferencia de Transformers (vs GPU que hace todo)
- **Ecosistema:** CUDA tiene 15 años de herramientas; ASICs requieren compiladores propios
- **Disponibilidad:** Solo vía cloud (Groq Cloud, Cerebras Cloud)

### El futuro: especialización

La tendencia es clara: **inferencia de producción se moverá a ASICs**

**Razones:**
1. **Eficiencia energética:** 5-10x más trabajo por watt
2. **Costo a escala:** A millones de chips, ASIC < GPU
3. **Latencia:** Crítica para aplicaciones interactivas

**Otros jugadores:**
- Google TPU v5/v6
- AWS Trainium/Inferentia
- Microsoft Maia
- Startups: Etched (Sohu), Tenstorrent, SambaNova

**Predicción:** En 5 años, >80% de la inferencia de producción correrá en ASICs, no en GPUs.

### Cuándo usar cada uno

| Caso de uso | Hardware recomendado |
|---|---|
| Entrenamiento de modelos | GPU (H100, A100) |
| Fine-tuning | GPU |
| Inferencia interactiva (chat) | **ASIC (Groq LPU)** |
| Generación de contenido largo | **ASIC (Cerebras WSE)** |
| Prototipado rápido | GPU (más flexible) |
| Producción a escala | **ASIC** (más eficiente) |
| Modelos multimodales complejos | GPU (más memoria) |
| Batch processing simple | **ASIC** (más rápido) |

In [ ]:
# Comparación de arquitecturas: GPU vs ASIC para inferencia
import matplotlib.pyplot as plt
import numpy as np

# Datos de comparación
arquitecturas = ['GPU\n(H100)', 'Groq\nLPU', 'Cerebras\nWSE']

# Métricas clave
sram_mb = [50, 230, 40000]
bandwidth_tbs = [3.35, 80, 20000]
throughput_toks = [65, 280, 2200]
ttft_ms = [350, 50, 100]

# Visualización 1: Speedup relativo a GPU
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Subplot 1: SRAM on-chip
ax = axes[0, 0]
speedup_sram = [s / sram_mb[0] for s in sram_mb]
bars = ax.bar(arquitecturas, speedup_sram, color=['steelblue', 'orange', 'green'], 
               alpha=0.8, edgecolor='white')
ax.set_ylabel('Speedup (GPU = 1x)')
ax.set_title('SRAM on-chip (más es mejor)', fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

for bar, val, orig in zip(bars, speedup_sram, sram_mb):
    label = f'{orig/1000:.0f}GB' if orig >= 1000 else f'{orig:.0f}MB'
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.2,
            f'{val:.0f}x\n{label}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Subplot 2: Bandwidth
ax = axes[0, 1]
speedup_bw = [b / bandwidth_tbs[0] for b in bandwidth_tbs]
bars = ax.bar(arquitecturas, speedup_bw, color=['steelblue', 'orange', 'green'], 
               alpha=0.8, edgecolor='white')
ax.set_ylabel('Speedup (GPU = 1x)')
ax.set_title('Bandwidth interno (más es mejor)', fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

for bar, val, orig in zip(bars, speedup_bw, bandwidth_tbs):
    label = f'{orig/1000:.0f}PB/s' if orig >= 1000 else f'{orig:.1f}TB/s'
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.2,
            f'{val:.0f}x\n{label}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Subplot 3: Throughput
ax = axes[1, 0]
speedup_throughput = [t / throughput_toks[0] for t in throughput_toks]
bars = ax.bar(arquitecturas, speedup_throughput, color=['steelblue', 'orange', 'green'], 
               alpha=0.8, edgecolor='white')
ax.set_ylabel('Speedup (GPU = 1x)')
ax.set_title('Throughput - Llama 70B (más es mejor)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val, orig in zip(bars, speedup_throughput, throughput_toks):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}x\n{orig} tok/s', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Subplot 4: Time-to-first-token (menor es mejor)
ax = axes[1, 1]
speedup_ttft = [ttft_ms[0] / t for t in ttft_ms]  # inverso porque menor es mejor
bars = ax.bar(arquitecturas, speedup_ttft, color=['steelblue', 'orange', 'green'], 
               alpha=0.8, edgecolor='white')
ax.set_ylabel('Speedup (GPU = 1x)')
ax.set_title('Velocidad de respuesta (más es mejor)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val, orig in zip(bars, speedup_ttft, ttft_ms):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{val:.1f}x\n{orig} ms', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('GPU vs ASIC para inferencia de LLMs\n(todos los speedups relativos a GPU H100)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Visualización 2: Utilización de GPU en prefill vs decode
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Prefill (matriz-matriz)
fases = ['Cómputo\n(FLOPS)', 'Memoria\n(bandwidth)']
utilizacion_prefill = [80, 60]
ax1.barh(fases, utilizacion_prefill, color=['green', 'orange'], alpha=0.8, edgecolor='white')
ax1.set_xlabel('Utilización (%)')
ax1.set_title('Fase PREFILL (procesar input)\nOperación: Matriz-Matriz', fontweight='bold')
ax1.set_xlim(0, 100)
ax1.grid(True, alpha=0.3, axis='x')

for i, val in enumerate(utilizacion_prefill):
    ax1.text(val + 2, i, f'{val}%', va='center', fontweight='bold')

# Decode (matriz-vector)
utilizacion_decode = [20, 95]
ax2.barh(fases, utilizacion_decode, color=['tomato', 'green'], alpha=0.8, edgecolor='white')
ax2.set_xlabel('Utilización (%)')
ax2.set_title('Fase DECODE (generar output)\nOperación: Matriz-Vector', fontweight='bold')
ax2.set_xlim(0, 100)
ax2.grid(True, alpha=0.3, axis='x')

for i, val in enumerate(utilizacion_decode):
    ax2.text(val + 2, i, f'{val}%', va='center', fontweight='bold')

plt.suptitle('Utilización de GPU en inferencia de LLMs\n(Decode es memory-bound → ASICs ganan)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("="*70)
print("RESUMEN DE SPEEDUPS (GPU H100 = baseline)")
print("="*70)
print(f"\n{'Arquitectura':<15} {'SRAM':>8} {'Bandwidth':>12} {'Throughput':>12} {'Latencia':>10}")
print("-"*70)
for i, arch in enumerate(['GPU (H100)', 'Groq LPU', 'Cerebras WSE']):
    print(f"{arch:<15} {speedup_sram[i]:>7.0f}x {speedup_bw[i]:>11.0f}x {speedup_throughput[i]:>11.1f}x {speedup_ttft[i]:>9.1f}x")

print("\n💡 Lecciones clave:")
print("   • GPU: Diseñada para entrenamiento (matriz-matriz)")
print("   • Inferencia decode: memory-bound (GPU ociosa ~80% del tiempo)")
print("   • LPU: 4.6x más SRAM → menos accesos a memoria externa")
print("   • WSE: 800x más SRAM → todo cabe on-chip")
print("   • Resultado: ASICs 4-34x más rápidos que GPU en inferencia")
print("   • Futuro: Inferencia de producción se moverá a ASICs")


---
<a id='23'></a>
## 23. RAG (Retrieval-Augmented Generation) — LLMs con conocimiento externo

**Definición:** **RAG (Retrieval-Augmented Generation)** es un patrón arquitectónico que combina **recuperación de información** (retrieval) con **generación de texto** (LLM) para que el modelo responda basándose en documentos externos, no solo en su conocimiento interno.

### El problema que resuelve RAG

**Sin RAG:**
- LLM solo sabe lo que vio durante el entrenamiento (cutoff date)
- Puede alucinar hechos que no conoce
- No puede acceder a información privada/propietaria

**Con RAG:**
- LLM accede a documentos actualizados en tiempo real
- Respuestas basadas en fuentes verificables
- Puede usar información privada (docs internos, bases de datos)

### Arquitectura de RAG

```
┌─────────────────────────────────────────────────────────┐
│               FASE 1: INDEXACIÓN (offline)              │
├─────────────────────────────────────────────────────────┤
│  Documentos → Chunking → Embeddings → Vector Database  │
└─────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────┐
│               FASE 2: QUERY (runtime)                   │
├─────────────────────────────────────────────────────────┤
│  Pregunta → Embedding → Similarity Search → Top-K docs │
│           → Inyectar en prompt → LLM → Respuesta        │
└─────────────────────────────────────────────────────────┘
```

### Componentes clave

**1. Document Loader**

Lee documentos de diversas fuentes:
- PDFs, Word, Excel
- Páginas web (scraping)
- Bases de datos (SQL, NoSQL)
- APIs

**2. Text Splitter (Chunking)**

Trocea documentos en fragmentos manejables:

| Estrategia | Descripción | Cuándo usar |
|---|---|---|
| Fixed-size | Chunks de tamaño fijo con overlap | Default razonable |
| Semantic | Trocea por cambios de tema | Documentos narrativos |
| Document-specific | Respeta estructura (headers, código) | Markdown, código |

**Parámetros clave:**
- `chunk_size`: Tamaño del chunk (ej. 500 caracteres)
- `chunk_overlap`: Overlap entre chunks (ej. 50 caracteres)

**3. Embeddings**

Convierte cada chunk a un vector:

```python
chunk = "RAG combina retrieval y generación"
embedding = embed_model.encode(chunk)  # → [0.1, -0.3, 0.7, ...]
```

**4. Vector Database**

Almacena embeddings para búsqueda rápida:

| Vector DB | Tipo | Cuándo usar |
|---|---|---|
| FAISS | Local | Prototipado, <1M docs |
| Chroma | Local/cloud | Apps pequeñas |
| Pinecone | Cloud managed | Producción, escala |
| Weaviate | Self-host | Control total |
| pgvector | Postgres | Si ya tienes Postgres |

**5. Retriever**

Busca los K chunks más relevantes:

```python
query = "¿Qué es RAG?"
query_embedding = embed_model.encode(query)
top_k_chunks = vector_db.similarity_search(query_embedding, k=4)
```

**6. Prompt Construction**

Inyecta los chunks recuperados en el prompt:

```
System: Eres un asistente. Responde usando SOLO el contexto dado.

Contexto:
[CHUNK 1]: RAG combina retrieval y generación...
[CHUNK 2]: Los embeddings permiten búsqueda semántica...
[CHUNK 3]: Vector databases indexan embeddings...

Pregunta: ¿Qué es RAG?
```

**7. LLM Generation**

El LLM genera la respuesta basándose en los chunks.

### Ejemplo completo

```python
# 1. Indexación (offline)
docs = load_documents("mi_documentacion/")
chunks = text_splitter.split_documents(docs)
embeddings = embed_model.embed_documents([c.text for c in chunks])
vector_db.add(chunks, embeddings)

# 2. Query (runtime)
query = "¿Cómo reseteo mi password?"
query_emb = embed_model.embed_query(query)
top_chunks = vector_db.similarity_search(query_emb, k=4)

prompt = f"""
Contexto: {format_chunks(top_chunks)}
Pregunta: {query}
"""

response = llm.generate(prompt)
```

### Métricas de calidad de RAG

**Retrieval metrics:**
- **Recall@K:** ¿Cuántos chunks relevantes están en los top-K?
- **Precision@K:** ¿Cuántos de los top-K son relevantes?
- **MRR (Mean Reciprocal Rank):** Posición del primer chunk relevante

**Generation metrics:**
- **Faithfulness:** ¿La respuesta se basa solo en los chunks?
- **Answer relevance:** ¿La respuesta responde la pregunta?
- **Context relevance:** ¿Los chunks son relevantes?

### Trade-offs de chunk size

| Chunk size | Ventajas | Desventajas |
|---|---|---|
| **Pequeño (200-300)** | Alta precisión, menos ruido | Contexto fragmentado |
| **Mediano (500-800)** | Balance | — |
| **Grande (1000-2000)** | Contexto completo | Más ruido, más caro |

### Optimizaciones avanzadas

**1. Hybrid search**

Combina búsqueda por keywords (BM25) + semántica (embeddings):

```
Score_final = 0.5 × Score_BM25 + 0.5 × Score_embedding
```

**2. Reranking**

Recupera muchos chunks (ej. 20), luego un modelo los reordena:

```
top_20 = vector_db.search(query, k=20)
top_4_reranked = reranker.rerank(query, top_20, k=4)
```

**3. Query rewriting**

Reescribe la pregunta del usuario para mejorar retrieval:

```
query_original = "cómo resetear pass"
query_reescrita = "¿Cómo puedo resetear mi contraseña?"
```

### Cuándo usar RAG

✅ **Usar RAG cuando:**
- Necesitas información actualizada (noticias, precios, docs)
- Información privada/propietaria (docs internos)
- Necesitas citar fuentes
- Reducir alucinaciones

❌ **NO usar RAG cuando:**
- El LLM ya sabe la respuesta (conocimiento general)
- Latencia es crítica (RAG agrega ~200-500ms)
- Presupuesto muy limitado (embeddings + retrieval + LLM)

In [ ]:
# Simulación de RAG (sin API, solo conceptual)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Simular documentos y embeddings
np.random.seed(42)

documentos = [
    "RAG combina retrieval y generación de texto",
    "Los embeddings son representaciones vectoriales",
    "Vector databases permiten búsqueda rápida",
    "El chunking divide documentos en fragmentos",
    "La pizza margarita lleva tomate y mozzarella",
    "Python es un lenguaje de programación",
    "Machine learning aprende de datos",
    "RAG reduce alucinaciones del LLM",
]

# Simular embeddings (en realidad serían de un modelo como text-embedding-3-small)
# Aquí los generamos de forma que docs similares tengan embeddings cercanos
embeddings_docs = np.random.randn(len(documentos), 3)
# Hacer que docs sobre RAG estén cerca
embeddings_docs[0] += [2, 0, 0]
embeddings_docs[7] += [2, 0, 0]
# Hacer que docs sobre embeddings/vectores estén cerca
embeddings_docs[1] += [0, 2, 0]
embeddings_docs[2] += [0, 2, 0]

print("="*70)
print("SIMULACIÓN DE RAG")
print("="*70)

# Query del usuario
query = "¿Qué es RAG y cómo funciona?"
query_embedding = np.array([2, 0, 0]) + np.random.randn(3) * 0.1  # Similar a docs de RAG

print(f"\nQuery: {query}\n")

# Calcular similitud coseno
similarities = cosine_similarity([query_embedding], embeddings_docs)[0]

# Ordenar por similitud
indices_ordenados = np.argsort(similarities)[::-1]

print("Ranking de documentos por similitud:")
print(f"{'Rank':<6} {'Score':<8} {'Documento'}")
print("-"*70)
for rank, idx in enumerate(indices_ordenados, 1):
    print(f"{rank:<6} {similarities[idx]:<8.3f} {documentos[idx]}")

# Top-K chunks para RAG
k = 3
top_k_indices = indices_ordenados[:k]

print(f"\n{'='*70}")
print(f"TOP-{k} CHUNKS RECUPERADOS (los que se pasan al LLM)")
print("="*70)
for i, idx in enumerate(top_k_indices, 1):
    print(f"[{i}] (score: {similarities[idx]:.3f}) {documentos[idx]}")

print(f"\n{'='*70}")
print("PROMPT CONSTRUIDO PARA EL LLM")
print("="*70)

contexto = "\n".join([f"[{i}] {documentos[idx]}" for i, idx in enumerate(top_k_indices, 1)])
prompt = f"""
System: Eres un asistente. Responde usando SOLO el contexto dado.

Contexto:
{contexto}

Pregunta: {query}
"""

print(prompt)

# Visualización 1: Embeddings en 3D
fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')

# Plotear documentos
for i, (emb, doc) in enumerate(zip(embeddings_docs, documentos)):
    color = 'red' if i in top_k_indices else 'lightgray'
    marker = 'o' if i in top_k_indices else 'x'
    size = 100 if i in top_k_indices else 30
    ax1.scatter(emb[0], emb[1], emb[2], c=color, marker=marker, s=size, alpha=0.7)
    if i in top_k_indices:
        ax1.text(emb[0], emb[1], emb[2], f'  Doc{i}', fontsize=8)

# Plotear query
ax1.scatter(query_embedding[0], query_embedding[1], query_embedding[2], 
           c='blue', marker='*', s=300, label='Query', edgecolors='black', linewidths=2)
ax1.text(query_embedding[0], query_embedding[1], query_embedding[2], '  Query', 
        fontsize=10, fontweight='bold')

ax1.set_xlabel('Dim 1')
ax1.set_ylabel('Dim 2')
ax1.set_zlabel('Dim 3')
ax1.set_title('Embeddings en espacio 3D\\n(rojo = top-3 recuperados)')
ax1.legend()

# Visualización 2: Similitud coseno
ax2 = fig.add_subplot(122)

colores = ['red' if i in top_k_indices else 'lightgray' for i in range(len(documentos))]
bars = ax2.barh(range(len(documentos)), similarities, color=colores, alpha=0.8, edgecolor='white')
ax2.set_yticks(range(len(documentos)))
ax2.set_yticklabels([f'Doc {i}' for i in range(len(documentos))])
ax2.set_xlabel('Similitud coseno')
ax2.set_title(f'Similitud de cada documento con la query\\n(top-{k} en rojo)')
ax2.grid(True, alpha=0.3, axis='x')
ax2.axvline(x=similarities[top_k_indices[-1]], color='red', linestyle='--', 
           label=f'Umbral top-{k}', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

# Análisis de chunk size
print("\n" + "="*70)
print("ANÁLISIS DE CHUNK SIZE")
print("="*70)

chunk_sizes = [200, 500, 1000, 2000]
doc_length = 5000  # caracteres

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Número de chunks
n_chunks = [doc_length / size for size in chunk_sizes]
ax1.bar(range(len(chunk_sizes)), n_chunks, color='steelblue', alpha=0.8, edgecolor='white')
ax1.set_xticks(range(len(chunk_sizes)))
ax1.set_xticklabels([f'{s}' for s in chunk_sizes])
ax1.set_xlabel('Chunk size (caracteres)')
ax1.set_ylabel('Número de chunks')
ax1.set_title('Documento de 5,000 caracteres')
ax1.grid(True, alpha=0.3, axis='y')

for i, val in enumerate(n_chunks):
    ax1.text(i, val + 0.5, f'{val:.0f}', ha='center', fontweight='bold')

# Subplot 2: Trade-offs
metrics = ['Precisión', 'Contexto', 'Costo']
small_chunk = [9, 3, 8]   # chunk pequeño: alta precisión, poco contexto, alto costo
large_chunk = [4, 9, 3]   # chunk grande: baja precisión, mucho contexto, bajo costo

x = np.arange(len(metrics))
width = 0.35

bars1 = ax2.bar(x - width/2, small_chunk, width, label='Chunk pequeño (200)', 
               color='tomato', alpha=0.8)
bars2 = ax2.bar(x + width/2, large_chunk, width, label='Chunk grande (2000)', 
               color='green', alpha=0.8)

ax2.set_ylabel('Score (1-10)')
ax2.set_title('Trade-offs de chunk size')
ax2.set_xticks(x)
ax2.set_xticklabels(metrics)
ax2.legend()
ax2.set_ylim(0, 10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n💡 Lecciones clave:")
print("   • RAG = Retrieval (buscar docs) + Augmented Generation (LLM con contexto)")
print("   • Similarity search encuentra los K chunks más cercanos a la query")
print("   • Solo los top-K se pasan al LLM (no todos los documentos)")
print("   • Chunk size pequeño: más precisión, menos contexto")
print("   • Chunk size grande: más contexto, menos precisión")
print("   • RAG reduce alucinaciones al basar respuestas en fuentes verificables")


---
<a id='1'></a>
## 1. Tokens y Tokenización

**Definición:** La **tokenización** es el proceso de dividir texto en unidades pequeñas llamadas **tokens**. Un token puede ser una palabra completa, una subpalabra, un carácter o incluso un byte.

### ¿Por qué no usar palabras completas?

| Problema | Solución con subpalabras |
|---|---|
| Vocabulario gigante (millones de palabras) | Vocabulario manejable (30k-200k tokens) |
| Palabras fuera del vocabulario (OOV) | Cualquier palabra se puede representar |
| No maneja morfología (correr, corriendo, corrió) | Comparte raíz: `[corr][iendo]`, `[corr][ió]` |

### Algoritmos principales

**1. Byte-Pair Encoding (BPE)** — usado por GPT, LLaMA
- Empieza con caracteres individuales
- Iterativamente fusiona los pares más frecuentes
- Ejemplo: `"running"` → `["run", "ning"]` o `["runn", "ing"]` según frecuencia

**2. WordPiece** — usado por BERT
- Similar a BPE pero maximiza la verosimilitud del corpus
- Usa `##` para marcar continuaciones: `["run", "##ning"]`

**3. SentencePiece** — usado por T5, LLaMA
- Trata el texto como secuencia de bytes (no requiere pre-tokenización)
- Soporta cualquier idioma sin espacios (chino, japonés)

### Propiedades importantes

- **Vocabulario fijo:** típicamente 30k-200k tokens
- **Reversible:** tokens → texto original (casi siempre)
- **Dependiente del modelo:** cada modelo tiene su propio tokenizer
- **Impacto en costo:** más tokens = más caro (APIs cobran por token)